In [1]:
import json

for split in ["preference_train.jsonl", "preference_val.jsonl"]:
    path = f"/home/mrb/projects/proj_2026_1/data/reward/{split}"
    zh_samples = []
    total = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            total += 1
            obj = json.loads(line)
            text = obj["chosen"][0]["content"]
            if any("\u4e00" <= c <= "\u9fff" for c in text):
                zh_samples.append(text)

    print(f"=== {split} ===")
    print(f"Total: {total}, Chinese: {len(zh_samples)}")
    for i, s in enumerate(zh_samples[:2]):
        print(f"  Sample {i+1}: {s[:80]}")
    print()

=== preference_train.jsonl ===
Total: 37800, Chinese: 12582
  Sample 1: 某家航空公司剛逢罷工，正在於是舉行特價優惠活動     如果太太跟先生一起乘坐商務艙，先生只需要為太太付半價的機票，該公司的顧客服務部門，希望得知顧客對此活動的
  Sample 2: 这哥们喝醉了，不小心一屁股坐到啤酒瓶上，正中菊花，然后我们几个扶他回宿舍了。第二天看他独自坐在床上，倚着墙，手捂着菊花抽着烟看着我们，任凭我们怎么解释都不说话…

=== preference_val.jsonl ===
Total: 4200, Chinese: 1422
  Sample 1: 有一天，老師發考卷給小明，叫小明拿回家給爸爸簽名。     隔天，老師問小明：「你爸爸說了什麼？」     小明：「老師，『髒話』的部份要去掉嗎？」     老
  Sample 2: 「有三個妙龄女郎在路上吃冰淇淋，一個用咬的，一個用舔的，第三個用吸的，請問哪個已經結婚？」     老師有點臉紅的回答：「啊是不是那個用吸的？」     學生詭



In [2]:
import sys
sys.path.insert(0, "/home/mrb/projects/proj_2026_1")

In [3]:
from rl.reward_model import build_reward_model_scorer

scorer = build_reward_model_scorer("MRBSTUDIO/Humor-Reward-Model-1.7B")

# 手工构造几个明显有区分度的例子
cases_score_5 = [
    # Chinese (3)
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我给未来的自己发了条消息：别焦虑——他秒回我说已经在焦虑这条消息。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我的人生像下载进度条，永远卡在99%，还提示“剩余时间：未知”。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我问时间能不能慢一点，它说可以，但账单会更快一点。"),

    # Spanish (3)
    ("No dar ninguna otra información.", "Abrí un gimnasio para vagos: la primera máquina es una excusa automática."),
    ("No dar ninguna otra información.", "Mi despertador no suena, negocia; cada mañana me ofrece cinco minutos más a cambio de mi dignidad."),
    ("No dar ninguna otra información.", "Dicen que el dinero no da felicidad, pero al menos compra Wi-Fi para buscarla."),

    # English (3)
    ("Please give me the joke only, no other text.", "I told my future self to relax, but he already had anxiety about this conversation."),
    ("Please give me the joke only, no other text.", "I opened a bakery for pessimists—everything is half-empty and slightly burnt."),
    ("Please give me the joke only, no other text.", "My password is “incorrect,” so whenever I forget it, the computer reminds me: “Your password is incorrect.”")
]


cases_score_3 = [
    # Chinese (3)
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我想早起，可是床对我太有吸引力。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我说要减肥，结果晚饭先反对了。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我计划周末学习，但电视剧抢先安排了时间。"),

    # Spanish (3)
    ("No dar ninguna otra información.", "Quise hacer dieta, pero la pizza tenía otros planes."),
    ("No dar ninguna otra información.", "Mi cama es mi lugar favorito para pensar en ir al trabajo."),
    ("No dar ninguna otra información.", "Intenté ahorrar dinero, pero el café ganó la discusión."),

    # English (3)
    ("Please give me the joke only, no other text.", "I started eating healthier, but my snacks keep finding me."),
    ("Please give me the joke only, no other text.", "My phone battery lasts longer than my motivation."),
    ("Please give me the joke only, no other text.", "I tried to be organized, but my desk disagreed.")
]


cases_score_1 = [
    # Chinese (3)
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我今天喝了水，然后把杯子放下。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "天气是天气，没有别的。"),
    ("我想听个笑话。 请只给出笑话，不要给出任何其他文本。", "我坐在椅子上，椅子也在那里。"),

    # Spanish (3)
    ("No dar ninguna otra información.", "La puerta está cerrada y eso es todo."),
    ("No dar ninguna otra información.", "Caminé por la calle y luego dejé de caminar."),
    ("No dar ninguna otra información.", "El vaso tiene agua y sigue teniendo agua."),

    # English (3)
    ("Please give me the joke only, no other text.", "I looked at the wall and it was there."),
    ("Please give me the joke only, no other text.", "The book is on the table and that is the story."),
    ("Please give me the joke only, no other text.", "I walked outside and then I walked back inside.")
]

print("-"*30)
print("Score 5")
for prompt, response in cases_score_5:
    score = scorer(prompt, response)
    print(f"{score:+.4f}  |  {response[:70]}")

print("-"*30)
print("Score 3")
for prompt, response in cases_score_3:
    score = scorer(prompt, response)
    print(f"{score:+.4f}  |  {response[:70]}")

print("-"*30)
print("Score 1")
for prompt, response in cases_score_1:
    score = scorer(prompt, response)
    print(f"{score:+.4f}  |  {response[:70]}")



adapter_config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading reward model base: Qwen/Qwen3-1.7B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-1.7B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading reward model adapter: MRBSTUDIO/Humor-Reward-Model-1.7B


adapter_model.safetensors:   0%|          | 0.00/69.8M [00:00<?, ?B/s]

  Reward model loaded: 1738.0M parameters on cuda
------------------------------
Score 5
+0.9986  |  我给未来的自己发了条消息：别焦虑——他秒回我说已经在焦虑这条消息。
+0.9985  |  我的人生像下载进度条，永远卡在99%，还提示“剩余时间：未知”。
+0.9984  |  我问时间能不能慢一点，它说可以，但账单会更快一点。
-0.9574  |  Abrí un gimnasio para vagos: la primera máquina es una excusa automáti
-0.9651  |  Mi despertador no suena, negocia; cada mañana me ofrece cinco minutos 
-0.5069  |  Dicen que el dinero no da felicidad, pero al menos compra Wi-Fi para b
+0.9465  |  I told my future self to relax, but he already had anxiety about this 
+0.9318  |  I opened a bakery for pessimists—everything is half-empty and slightly
+0.9710  |  My password is “incorrect,” so whenever I forget it, the computer remi
------------------------------
Score 3
+0.9919  |  我想早起，可是床对我太有吸引力。
+0.9980  |  我说要减肥，结果晚饭先反对了。
+0.9972  |  我计划周末学习，但电视剧抢先安排了时间。
-0.9946  |  Quise hacer dieta, pero la pizza tenía otros planes.
-0.9848  |  Mi cama es mi lugar favorito para pensar en ir al trabajo.
-0.9952  |  Intenté